In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
import os
import time
import imageio
import matplotlib.pyplot as plt
import numpy as np
from IPython import display
from tensorflow.keras import layers

# Set parameters
IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 1
BATCH_SIZE = 32
LATENT_DIM = 256
EPOCHS = 500
BUFFER_SIZE = 1000

# Path configurations
drive_path = "/content/drive/MyDrive"
data_dir = os.path.join(drive_path, "chest_xray/train/NORMAL")
save_dir = os.path.join(drive_path, "GAN_Models_Normal")
output_dir = os.path.join(drive_path, "Generated_Xrays_Normal")

# Create directories
os.makedirs(save_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# Load and preprocess dataset
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    directory=os.path.dirname(data_dir),  # Parent directory containing NORMAL/PNEUMONIA
    label_mode=None,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    shuffle=True
).map(lambda x: (x / 127.5) - 1.0)  # Normalize to [-1, 1]

# Improved Generator for 128x128 images
def make_generator():
    model = tf.keras.Sequential()
    model.add(layers.Dense(8*8*512, use_bias=False, input_shape=(LATENT_DIM,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    model.add(layers.Reshape((8, 8, 512)))

    # Upsample to 16x16
    model.add(layers.Conv2DTranspose(256, (5,5), strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # Upsample to 32x32
    model.add(layers.Conv2DTranspose(128, (5,5), strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # Upsample to 64x64
    model.add(layers.Conv2DTranspose(64, (5,5), strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # Upsample to 128x128
    model.add(layers.Conv2DTranspose(32, (5,5), strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # Final layer
    model.add(layers.Conv2DTranspose(1, (5,5), strides=1, padding='same', use_bias=False, activation='tanh'))

    return model

# Improved Discriminator
def make_discriminator():
    model = tf.keras.Sequential()
    model.add(layers.Conv2D(64, (5,5), strides=2, padding='same',
                                     input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5,5), strides=2, padding='same'))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(256, (5,5), strides=2, padding='same'))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

# Initialize models
generator = make_generator()
discriminator = make_discriminator()

# Loss functions and optimizers
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-5, beta_1=0.5)

# Checkpoint setup
checkpoint_prefix = os.path.join(save_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

# Training parameters
num_examples_to_generate = 9
seed = tf.random.normal([num_examples_to_generate, LATENT_DIM])

# Training functions
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, LATENT_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(9, 9))

    for i in range(predictions.shape[0]):
        plt.subplot(3, 3, i+1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')

    plt.savefig(os.path.join(output_dir, f'xray_epoch_{epoch:04d}_normal.png'))
    plt.close()

def train(dataset, epochs):
    # Try to restore latest checkpoint
    latest = tf.train.latest_checkpoint(save_dir)
    if latest:
        checkpoint.restore(latest)
        start_epoch = int(latest.split('-')[-1].split('.')[0])
        print(f"Resuming from epoch {start_epoch}")
    else:
        start_epoch = 0

    for epoch in range(start_epoch, epochs):
        start = time.time()

        for image_batch in dataset:
            gen_loss, disc_loss = train_step(image_batch)

        # Generate and save images every epoch
        display.clear_output(wait=True)
        generate_and_save_images(generator, epoch + 1, seed)

        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)
            print(f'Saved checkpoint at epoch {epoch+1}')

        print(f'Epoch {epoch+1} completed in {time.time()-start:.2f}s')
        print(f'Generator loss: {gen_loss:.4f}, Discriminator loss: {disc_loss:.4f}\n')

    # Generate final images
    display.clear_output(wait=True)
    generate_and_save_images(generator, epochs, seed)

# Start training
train(train_dataset, EPOCHS)

# Create GIF
anim_file = os.path.join(drive_path, 'xray_generation_normal.gif')

with imageio.get_writer(anim_file, mode='I') as writer:
    filenames = sorted(glob.glob(os.path.join(output_dir, 'xray_epoch_*.png')))
    last_image = None
    for filename in filenames:
        image = imageio.imread(filename)
        if image != last_image:  # Skip duplicate frames
            writer.append_data(image)
        last_image = image
    # Add last frame 10 times to pause at end
    for _ in range(10):
        writer.append_data(image)

display.display(display.HTML(f'<img src="{anim_file}">'))

Epoch 46 completed in 609.89s
Generator loss: 0.7283, Discriminator loss: 1.3265



KeyboardInterrupt: 